# A Stochastic Policy and Its Gradient

Turn observations into action probabilities and derive an imitation update by hand.

## Before You Start

Course 12. Entry course: 10. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** gradients, softmax-probabilities.

## What You Will Be Able to Do

- Compute a sigmoid decision boundary
- Differentiate L1 imitation loss
- Explain exploration and rollout shift

## Actions are sampled, not differentiable decisions

A binary policy sets p(action=1|s)=sigmoid(w dot s+b). Sampling chooses 1 with probability p and 0 otherwise. The p=0.5 decision boundary is w dot s+b=0. For CartPole the observation has cart position, cart velocity, pole angle, and angular velocity; the actions apply left or right force. A stochastic policy can explore alternatives, but randomness alone does not solve sparse rewards.

Imitation learning trains on labeled expert actions. For binary label y, L1 loss |p-y| equals 1 minus the probability assigned to the demonstrated action. Its derivative with respect to the logit is (1-2y)*p*(1-p). Multiply by each feature to obtain the weight gradient and average over examples. This differs from binary cross-entropy, whose logit derivative is p-y. Do not silently substitute one loss for the other.

## The deployed policy changes its own data

An imitation dataset comes from the expert's visited states. A small mistake can take the learned policy to a state absent from that dataset, where another mistake becomes more likely. This is rollout distribution shift. Evaluate complete trajectories, not only offline action accuracy. Reinforcement learning instead uses rewards from the learner's interaction, introducing exploration, variance, and credit-assignment problems.

The lab returns probabilities, mean L1 loss, and gradients for a batch. The notebook verifies them against autograd and finite differences. It also compares demonstration action accuracy with rollout reward in a deliberately small discrete task; the CartPole unit uses the real Gymnasium simulator. None of these tests authorizes control of physical hardware.

## Worked Example: One sigmoid update

s=[2,-1], w=[0,0], b=0, expert action y=1.

1. **Forward:** The logit is 0, p=.5, and L1=.5.
2. **Backward:** dL/dz=-.25. Thus dL/dw=[-.5,.25], dL/db=-.25.
3. **SGD:** At learning rate .1, w becomes [.05,-.025] and b becomes .025. The demonstrated action becomes more likely.

Differentiate the probability-based loss, not the sampled action.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Binary sigmoid policy and L1 imitation gradients."""
import numpy as np

def imitation_l1(states, actions, weights, bias):
    """Return dict probabilities, loss, grad_w, grad_b.

states finite [N,D], binary actions [N], weights [D], scalar bias; N>0.
Mean L1 between sigmoid probabilities and binary expert actions.
Assume valid shapes/values. Compute in floating point, including for
integer inputs. Stable at extreme logits; no input mutation."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

A demonstrated action is 0 and p(action=1)=0.8.

- L1 imitation loss

<details><summary>Worked answers</summary>

- 0.8: |.8-0|=.8; chosen-action confidence is .2.

</details>

## Experiment

Run `experiments/rl_gradients.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- Compare hand gradients with central differences and autograd before training.
- The discrete rollout-shift example is not a CartPole physics substitute.

## Transfer Problems

1. Derive the cross-entropy gradient and compare saturation with L1.
2. Plot the boundary in pole-angle/angular-velocity space with other observations fixed.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[A Reduction of Imitation Learning and Structured Prediction](https://arxiv.org/abs/1011.0686). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.



## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Binary sigmoid policy and L1 imitation gradients."""
import numpy as np

def imitation_l1(states, actions, weights, bias):
    """Return dict probabilities, loss, grad_w, grad_b.

states finite [N,D], binary actions [N], weights [D], scalar bias; N>0.
Mean L1 between sigmoid probabilities and binary expert actions.
Assume valid shapes/values. Compute in floating point, including for
integer inputs. Stable at extreme logits; no input mutation."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
result = imitation_l1(np.array([[2., -1.]]), np.array([1]), np.zeros(2), 0.)
assert result['loss'] == .5 and np.allclose(result['grad_w'], [-.5, .25]) and result['grad_b'] == -.25
unsigned_actions = imitation_l1(np.array([[2., -1.]]), np.array([1], dtype=np.uint8), np.zeros(2), 0.)
assert np.array_equal(unsigned_actions['probabilities'], [.5])
assert unsigned_actions['loss'] == .5 and np.array_equal(unsigned_actions['grad_w'], [-.5, .25]) and unsigned_actions['grad_b'] == -.25, 'Unsigned binary actions must have a negative imitation slope'
for states, actions, weights, bias in [
    (np.array([[200]], dtype=np.uint8), np.array([1], dtype=np.uint8), np.array([2], dtype=np.uint8), np.int16(-400)),
    (np.array([[100, 100]], dtype=np.int8), np.array([0], dtype=np.uint8), np.array([2, -1], dtype=np.int8), np.int16(-100)),
    (np.array([[-100]], dtype=np.int8), np.array([1], dtype=np.int8), np.array([2], dtype=np.int8), np.int16(200)),
]:
    before = [value.copy() for value in (states, actions, weights)]
    result = imitation_l1(states, actions, weights, bias)
    slope = .25 if actions[0] == 0 else -.25
    assert np.array_equal(result['probabilities'], [.5]) and result['loss'] == .5
    assert np.array_equal(result['grad_w'], states.astype(float)[0] * slope) and result['grad_b'] == slope, 'Promote integer operands before matmul and signed cancellation'
    assert result['probabilities'].dtype.kind == 'f' and result['grad_w'].dtype.kind == 'f'
    assert isinstance(result['loss'], float) and isinstance(result['grad_b'], float)
    assert all(np.array_equal(value, original) for value, original in zip((states, actions, weights), before))
rng = np.random.default_rng(17)
x, y, w = rng.normal(size=(7, 3)), rng.integers(2, size=7), rng.normal(size=3)
b = .3; got = imitation_l1(x, y, w, b)
def loss(weight, bias):
    p = 1/(1+np.exp(-(x@weight+bias)))
    return np.abs(p-y).mean()
eps = 1e-5
for i in range(3):
    delta = np.eye(3)[i]*eps
    assert np.isclose(got['grad_w'][i], (loss(w+delta,b)-loss(w-delta,b))/(2*eps))
assert np.isclose(got['grad_b'], (loss(w,b+eps)-loss(w,b-eps))/(2*eps))
assert np.isfinite(imitation_l1(np.array([[1000.], [-1000.]]), np.array([0, 1]), np.ones(1), 0.)['loss'])
print('Sigmoid imitation checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Binary sigmoid policy and L1 imitation gradients."""
import numpy as np


def imitation_l1(states, actions, weights, bias):
    """Return dict probabilities, loss, grad_w, grad_b.

    states finite [N,D], binary actions [N], weights [D], scalar bias; N>0.
    Mean L1 between sigmoid probabilities and binary expert actions.
    Assume valid shapes/values. Compute in floating point, including for
    integer inputs. Stable at extreme logits; no input mutation.
    """
    states, actions, weights = [np.asarray(value, float) for value in (states, actions, weights)]
    bias = float(bias)
    logits = states @ weights + bias
    p = np.exp(-np.logaddexp(0., -logits))
    slope = (1-2*actions)*p*(1-p)
    return dict(probabilities=p, loss=float(np.abs(p-actions).mean()), grad_w=states.T@slope/len(states), grad_b=float(slope.mean()))


In [ ]:
import numpy as np
result = imitation_l1(np.array([[2., -1.]]), np.array([1]), np.zeros(2), 0.)
assert result['loss'] == .5 and np.allclose(result['grad_w'], [-.5, .25]) and result['grad_b'] == -.25
unsigned_actions = imitation_l1(np.array([[2., -1.]]), np.array([1], dtype=np.uint8), np.zeros(2), 0.)
assert np.array_equal(unsigned_actions['probabilities'], [.5])
assert unsigned_actions['loss'] == .5 and np.array_equal(unsigned_actions['grad_w'], [-.5, .25]) and unsigned_actions['grad_b'] == -.25, 'Unsigned binary actions must have a negative imitation slope'
for states, actions, weights, bias in [
    (np.array([[200]], dtype=np.uint8), np.array([1], dtype=np.uint8), np.array([2], dtype=np.uint8), np.int16(-400)),
    (np.array([[100, 100]], dtype=np.int8), np.array([0], dtype=np.uint8), np.array([2, -1], dtype=np.int8), np.int16(-100)),
    (np.array([[-100]], dtype=np.int8), np.array([1], dtype=np.int8), np.array([2], dtype=np.int8), np.int16(200)),
]:
    before = [value.copy() for value in (states, actions, weights)]
    result = imitation_l1(states, actions, weights, bias)
    slope = .25 if actions[0] == 0 else -.25
    assert np.array_equal(result['probabilities'], [.5]) and result['loss'] == .5
    assert np.array_equal(result['grad_w'], states.astype(float)[0] * slope) and result['grad_b'] == slope, 'Promote integer operands before matmul and signed cancellation'
    assert result['probabilities'].dtype.kind == 'f' and result['grad_w'].dtype.kind == 'f'
    assert isinstance(result['loss'], float) and isinstance(result['grad_b'], float)
    assert all(np.array_equal(value, original) for value, original in zip((states, actions, weights), before))
rng = np.random.default_rng(17)
x, y, w = rng.normal(size=(7, 3)), rng.integers(2, size=7), rng.normal(size=3)
b = .3; got = imitation_l1(x, y, w, b)
def loss(weight, bias):
    p = 1/(1+np.exp(-(x@weight+bias)))
    return np.abs(p-y).mean()
eps = 1e-5
for i in range(3):
    delta = np.eye(3)[i]*eps
    assert np.isclose(got['grad_w'][i], (loss(w+delta,b)-loss(w-delta,b))/(2*eps))
assert np.isclose(got['grad_b'], (loss(w,b+eps)-loss(w,b-eps))/(2*eps))
assert np.isfinite(imitation_l1(np.array([[1000.], [-1000.]]), np.array([0, 1]), np.ones(1), 0.)['loss'])
print('Sigmoid imitation checks passed.')


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""Exact expected-return gradient versus Monte Carlo and finite differences."""
from itertools import product
import json
import numpy as np
import torch


def gradient_experiment():
    states = torch.tensor([[1., -1.], [.5, 2.]], dtype=torch.float64)
    theta = torch.tensor([.3, -.2], dtype=torch.float64, requires_grad=True)
    labels = torch.tensor([1., 0.], dtype=torch.float64)
    p = torch.sigmoid(states@theta)
    imitation = (p-labels).abs().mean()
    imitation.backward()
    hand = states.T@((1-2*labels)*p.detach()*(1-p.detach()))/2
    assert torch.allclose(theta.grad, hand)
    theta.grad = None

    def expected_return(weight):
        probabilities = torch.sigmoid(states@weight)
        total = torch.zeros((), dtype=torch.float64)
        for actions in product([0, 1], repeat=2):
            action = torch.tensor(actions)
            likelihood = torch.where(action.bool(), probabilities, 1-probabilities).prod()
            reward = float(actions == (1, 0))
            total = total + likelihood*reward
        return total
    objective = expected_return(theta); objective.backward()
    exact = theta.grad.detach().numpy()
    finite = []
    for i in range(2):
        delta = torch.eye(2, dtype=torch.float64)[i]*1e-5
        finite.append(float(((expected_return(theta.detach()+delta)-expected_return(theta.detach()-delta))/(2e-5))))
    assert np.allclose(exact, finite, atol=1e-8)
    rng = np.random.default_rng(8); prob = torch.sigmoid(states@theta.detach()).numpy()
    actions = (rng.random((20000, 2)) < prob).astype(float)
    reward = ((actions[:, 0] == 1) & (actions[:, 1] == 0)).astype(float)
    scores = (actions-prob)@states.numpy()
    estimates = reward[:, None]*scores
    baseline_estimates = (reward-float(objective.detach()))[:, None]*scores
    # A simple compounding-error model: independent local correctness q over H steps.
    rollout = [{'offline_action_accuracy': q, 'horizon': h, 'all_actions_correct_probability': q**h} for q in [.9,.99] for h in [5,50]]
    return dict(exact_gradient=exact.tolist(), finite_difference=finite,
        monte_carlo=estimates.mean(0).tolist(), samples=len(actions),
        raw_variance=estimates.var(0).tolist(), baseline_variance=baseline_estimates.var(0).tolist(),
        imitation_gradient=hand.tolist(), rollout_shift_toy=rollout,
        limitation='The q^H toy assumes independent local errors; real distribution shift can be more complex.')


if __name__ == '__main__':
    print(json.dumps(gradient_experiment(), indent=2))
